In [23]:
!pip install youtube-comment-downloader

import pandas as pd
from itertools import islice
from youtube_comment_downloader import YoutubeCommentDownloader

url = "https://www.youtube.com/watch?v=ZvbifV4mKb0"
comments = YoutubeCommentDownloader().get_comments_from_url(url)

rows = [c["text"] for c in islice(comments, 100)]
df = pd.DataFrame({"comment": rows, "label": ""})
df = df.drop_duplicates(subset="comment")
df.to_csv("comments.csv", index=False)
print(df.shape)
df.head()

(96, 2)


,comment,label
0,Don't remove the discs !,
1,Remember: this game STILL MANAGED TO SELL 22.9...,
2,💿💿💿💿💿💿💿💿,
3,Sony playstation really become a JOKE!! like s...,
4,That was more exciting than playing the whole ...,


In [ ]:
import pandas as pd

df = pd.read_csv("comments.csv")
df["label"] = df["label"].fillna("")
options = {"p": "Positive", "n": "Negative", "u": "Neutral"}

for i in range(len(df)):
    if df.loc[i, "label"] != "":
        continue
    print(f"\n[{i}] {df.loc[i, 'comment']}")
    key = input("p / n / u / x (skip) / s (stop): ").lower()
    if key == "s":
        break
    if key == "x":
        df.loc[i, "label"] = "SKIP"
        continue
    df.loc[i, "label"] = options.get(key, "Neutral")
    df.to_csv("comments_labeled.csv", index=False)

df.to_csv("comments_labeled.csv", index=False)
print(df["label"].value_counts())

In [50]:
#let's build the model
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv("comments_labeled.csv")
df = df[df["label"].isin(["Positive", "Negative", "Neutral"])]
print(df["label"].value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    df["comment"], df["label"],
    test_size=0.2, stratify=df["label"], random_state=42)

model = make_pipeline(
    TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2),
    LogisticRegression(max_iter=1000, class_weight="balanced"))
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

label
Negative    57
Neutral     20
Positive     6
Name: count, dtype: int64
Accuracy: 0.588
              precision    recall  f1-score   support

    Negative       0.69      0.75      0.72        12
     Neutral       0.25      0.25      0.25         4
    Positive       0.00      0.00      0.00         1

    accuracy                           0.59        17
   macro avg       0.31      0.33      0.32        17
weighted avg       0.55      0.59      0.57        17

[[9 3 0]
 [3 1 0]
 [1 0 0]]


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [27]:
import joblib
joblib.dump(model,'sentiment_model.pkl')
print('saved')

saved


In [28]:
import joblib
loaded_model=joblib.load('sentiment_model.pkl')

In [29]:
def predict_sentiment(comment):
    if not comment or not isinstance(comment,str) or not comment.strip():
        return "Error: input must be a non-empty string"

    prediction=loaded_model.predict([comment])
    return prediction[0]


In [30]:
comments=[
    'Sony is going down',
    '',
    'The game is good',
    'Worst gameplay',
    '100',
]

In [31]:
for comment in comments:
    print(f"Input:{comment!r}--> Prediction:{predict_sentiment(comment)!r}")

Input:'Sony is going down'--> Prediction:'Negative'
Input:''--> Prediction:'Error: input must be a non-empty string'
Input:'The game is good'--> Prediction:'Positive'
Input:'Worst gameplay'--> Prediction:'Negative'
Input:'100'--> Prediction:'Negative'


In [41]:
#Task 3
#moer evaluatiion examples

eval_examples=[
    ('This trailer looks incredible!','Positive'),
    ('Bring back physical games!','Negative'),
    ('5/10','Neutral'),
]

for comment,expected in eval_examples:
  predicted=predict_sentiment(comment)
  match="✅" if predicted==expected else "❌"
  print(f"{match} Input:{comment!r} | Expected: {expected!r} | Predicted:{predicted!r}")

✅ Input:'This trailer looks incredible!' | Expected: 'Positive' | Predicted:'Positive'
✅ Input:'Bring back physical games!' | Expected: 'Negative' | Predicted:'Negative'
✅ Input:'5/10' | Expected: 'Neutral' | Predicted:'Neutral'


In [42]:
#Edge Cases

edge_cases=[
    "ok",
    "yeah right, amazing launch",
    "SONY SONY SONY",
    "😍😍😍",
    "Not bad, not great either",
]


for comment in edge_cases:
  print(f"Input:{comment!r}->Predicted:{predict_sentiment(comment)!r}")


Input:'ok'->Predicted:'Negative'
Input:'yeah right, amazing launch'->Predicted:'Negative'
Input:'SONY SONY SONY'->Predicted:'Negative'
Input:'😍😍😍'->Predicted:'Negative'
Input:'Not bad, not great either'->Predicted:'Negative'


In [63]:
#Failure cases
df_test=pd.DataFrame({'comment':X_test,'actual':y_test,'predicted':pred})
failures=df_test[df_test['actual']!=df_test['predicted']]
print(failures.to_string(index=False))


                                                                                                                                                                                                 comment   actual predicted
                                                                                                                         "play has no limits". If you follow the straight line that it does have limits. Positive  Negative
                                      This game plays itself lol.  Pass.  Also, Playstation, your recent decisions the last few years have been awful.  Thank you for destroying the once beloved brand.  Neutral  Negative
Question.... will Playstation end it's self with forced diversity hand holding trash games before it ends itself with digital only dynamic pricing? \nAt this rate it won't last to see dynamic pricing?  Neutral  Negative
                                                                                                             I bought th

In [68]:
#A simple interface
import ipywidgets as widgets
from IPython.display import display
text_box=widgets.Text(placeholder='Type a comment...')
button=widgets.Button(description='Predict')
output=widgets.Output()

def on_click(b):
  output.clear_output()
  with output:
   print(predict_sentiment(text_box.value))

button.on_click(on_click)
display(text_box,button,output)

Text(value='', placeholder='Type a comment...')

Button(description='Predict', style=ButtonStyle())

Output()